## GPR on WeatherBench Data - z500
Apply GP regression baseline (using exact GP to train on monthly data)

In [1]:
# Check GPU
!nvidia-smi

%load_ext autoreload
%autoreload 2

Sun Apr  6 15:22:21 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.12             Driver Version: 535.104.12   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          Off | 00000000:1B:00.0 Off |                    0 |
| N/A   48C    P0              64W / 300W |  32451MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import time
import gc
from tqdm.notebook import tqdm

import torch
import gpytorch

from GPyTorch.GPRs.model import ExactGP
from GPyTorch.GPRs.utils import splitting_data, extreme_points_rel_err, extreme_points_rmse
from GPyTorch.GPRs.process_data import create_data
from GPyTorch.GPRs.plot import *

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Defining the Kernel

We use a separable kernel in space and time:
$$K(x,y,z,t) = K_{\text{space}}(x,y,z) \times K_{\text{time}}(t),$$
where $K_{\text{space}}(x,y,z) = \sum_n K_{n\text{, Matern}}(x,y,z)$ and $K_{\text{time}}(t) = \sum_n K_{n\text{, Periodic}}(t)$.

In [3]:
# Define the GP regression kernel, separable in space and time
### First define the spatial kernel 
spatial_lengths = [0.08, 0.25, 0.5]

# Smoothness param of matern kernel set to 3/2
matern_spatial_1 = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[0,1,2]).to(device)
matern_spatial_1.lengthscale = torch.tensor(spatial_lengths[0]).to(device)
matern_spatial_2 = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[0,1,2]).to(device)
matern_spatial_2.lengthscale = torch.tensor(spatial_lengths[1]).to(device)
matern_spatial_3 = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[0,1,2]).to(device)
matern_spatial_3.lengthscale = torch.tensor(spatial_lengths[2]).to(device)

spatial_kernel = matern_spatial_1 + matern_spatial_2 + matern_spatial_3

In [4]:
### For temporal kernel
period_lengths = [24, 24*30*3, 24*30*12]

time_kernel1 = gpytorch.kernels.PeriodicKernel(active_dims=[3]).to(device) # Daily    
time_kernel1.period_length = torch.tensor(period_lengths[0]).to(device)
time_kernel2 = gpytorch.kernels.PeriodicKernel(active_dims=[3]).to(device) 
time_kernel2.period_length = torch.tensor(period_lengths[1]).to(device)
time_kernel3 = gpytorch.kernels.PeriodicKernel(active_dims=[3]).to(device) 
time_kernel3.period_length = torch.tensor(period_lengths[2]).to(device)

matern_time_1 = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[3]).to(device)
#matern_time_1.lengthscale = torch.tensor(40).to(device)

time_kernel = matern_time_1 + time_kernel1 + time_kernel2 + time_kernel3

In [5]:
# Assembling the Kernel

# If wrap scaling only to the spatial kernel
# spatial_kernel = gpytorch.kernels.ScaleKernel(spatial_kernel).to(device)
# kernel = spatial_kernel * time_kernel

# If wrap scaling to the whole kernel
kernel = gpytorch.kernels.ScaleKernel(spatial_kernel * time_kernel).to(device)

## Training and Evaluation

In [ ]:
def train_gp(lead_time_h, data_train, data_test, space_subsample=1, time_subsample=1, frac=0.1, epochs=100, save_model=False, device=device):
    """Train a Gaussian Process model using GPyTorch. Can add subsampling the data for efficiency."""

    # Prepare training data
    X_train, Y_train = create_data(data_train, data_test, lead_time_h, space_subsample=space_subsample, 
                                   time_subsample=time_subsample)
    X_train = X_train.to(device)
    Y_train = Y_train.to(device)
    #X_train = X_train.view(-1, 4)

    # Random Sampling 

    n_frac = 10000
    N = X_train.shape[0] * X_train.shape[1] * X_train.shape[2] 
    # n_frac = int(frac * N) # Fraction of points to be chosen
    X_train = X_train.view(-1, 4)
    indices = torch.randperm(X_train.size(0))[:n_frac]
    X_train = X_train[indices] 
    Y_train = Y_train[indices]

    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    #likelihood.noise = torch.tensor(1e-4).to(device)  
    #likelihood.noise_covar.raw_noise.requires_grad_(False) 
    model = ExactGP(X_train, Y_train, likelihood, kernel).to(device)
    
    # Train model
    model.train()
    likelihood.train()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005) 
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.965)

    # Loss function
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    # Training loop
    print("Starting Exact GP regression training:")
    # Reset the peak memory statistics before training
    torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()

    hyperparams_track = []
    with tqdm(total=epochs, desc="Training Progress", unit="epoch") as pbar:
        for i in range(epochs):
            optimizer.zero_grad()
            output = model(X_train)
            loss_val = -mll(output, Y_train)
            loss_val.backward()
            optimizer.step() 
            scheduler.step()
            pbar.set_postfix(loss=f"{loss_val.item():.3f}")
            pbar.update(1)

            # Record model params
            spatial_kernel = model.covar_module.base_kernel.kernels[0]
            temporal_kernel = model.covar_module.base_kernel.kernels[1]
            hyperparams_track.append({
                "loss": loss_val.item(),
                "rmse": torch.sqrt(torch.mean((output.mean - Y_train)**2)).item(),
                "mean": model.mean_module.constant.item(),
                "noise": likelihood.noise.item(),
                "outputscale": model.covar_module.outputscale.item(),
                "spatial_1_lengthscale": spatial_kernel.kernels[0].lengthscale.item(),
                "spatial_2_lengthscale": spatial_kernel.kernels[1].lengthscale.item(),
                "spatial_3_lengthscale": spatial_kernel.kernels[2].lengthscale.item(),
                "time_matern_lengthscale": temporal_kernel.kernels[0].lengthscale.item(),
                "time_lengthscale_1": temporal_kernel.kernels[1].lengthscale.item(),
                "time_period_1": temporal_kernel.kernels[1].period_length.item(),
                "time_lengthscale_2": temporal_kernel.kernels[2].lengthscale.item(),
                "time_period_2": temporal_kernel.kernels[2].period_length.item(),
                "time_lengthscale_3": temporal_kernel.kernels[3].lengthscale.item(),
                "time_period_3": temporal_kernel.kernels[3].period_length.item(),
            })
            
    current_allocated = torch.cuda.memory_allocated(device)
    max_allocated = torch.cuda.max_memory_allocated(device)
    print(f"Current GPU memory allocated: {current_allocated / (1024**2):.2f} MB")
    print(f"Peak GPU memory allocated: {max_allocated / (1024**2):.2f} MB")

    end_time = time.time()
    print("Training completed")
    time_elapsed = end_time - start_time
    print(f"Training time: {time_elapsed:.2f}s")

    # Save model
    if save_model:
        torch.save(model.state_dict(), "saved_models/gp_model.pth")
        torch.save(likelihood.state_dict(), "saved_models/gp_likelihood.pth")
        print("Saved model")

    return model, likelihood, hyperparams_track, X_train, Y_train

In [7]:
def predictions(model, likelihood, data_train, data_test, data_std, data_mean, lead_time, space_subsample, time_subsample):
    """Producing predictions after training"""

    X_test, Y_test, lat, lon = create_data(data_train, data_test, lead_time, space_subsample=space_subsample, 
                                                time_subsample=time_subsample, train=False)
    X_test = X_test.to(device)
    Y_test = Y_test.to(device)
    X_test = X_test.view(-1,4)
    nlat = len(lat)
    nlon = len(lon)

    # Make Predictions
    model.eval()
    likelihood.eval()
    with torch.no_grad():
        pred_dist = model(X_test)
        Y_pred = pred_dist.mean
        Y_std = pred_dist.stddev  
        mse = torch.mean((Y_pred - Y_test)**2)
        rmse = torch.sqrt(mse)
        print("MSE: ", mse.item())
        print("RMSE: ", rmse.item())

        Y_pred_actual = (Y_pred.detach().cpu().numpy() * data_std.values + data_mean.values).reshape(-1,nlat,nlon)
        Y_test_actual = (Y_test.detach().cpu().numpy() * data_std.values + data_mean.values).reshape(-1,nlat,nlon)
        err = Y_pred_actual - Y_test_actual
        weights_lat = np.cos(np.deg2rad(lat))
        weights_lat /= weights_lat.mean()
        wrmse = np.sqrt((err**2 * weights_lat[None, :, None]).mean())
        print("Weighted RMSE: ", wrmse)

    Y_pred = Y_pred.detach().cpu().numpy()  
    Y_test = Y_test.detach().cpu().numpy()  
    Y_std = Y_std.detach().cpu().numpy()

    return Y_pred_actual, Y_test_actual, Y_pred, Y_test, Y_std, lat, lon, wrmse.item()

## Iteratively applying GP regression on the all of the grids

We first split the full data (i.e. 32*64) into smaller subgrids, then fit a gp on each of the grid. We split by looking at the avg of the data.

In [8]:
z500 = xr.open_mfdataset('data/5.625deg/geopotential_500/*.nc', combine='by_coords').sel(time=slice('2000', '2018'))
lat_bound, lon_bound = splitting_data(z500, 'z')
print("lat. boundaries:", lat_bound)
print("lon. boundaries", lon_bound)

# Manually setting the boundaries
lat_bound = [0,13,18,32]
lon_bound = [0,5,8,22,43,50,60,64]

nlat = z500.lat.size
nlon = z500.lon.size
num_lat_bounds = len(lat_bound) - 1  
num_lon_bounds = len(lon_bound) - 1  
# To reconstruct the full error grid later
full_relative_error_grid = np.full((nlat, nlon), np.nan)
full_rmse_error_grid = np.full((nlat, nlon), np.nan)
wrmse_full = []

Peaks (lat indices): [13 18] Troughs (lat indices): [15]
Peaks (lon indices): [ 8 42 60] Troughs (lon indices): [ 5 22 50 62]
lat. boundaries: [ 0 13 15 18 32]
lon. boundaries [ 0  5  8 22 42 50 60 62 64]


In [ ]:
for i in range(num_lat_bounds):
    for j in range(num_lon_bounds):
        lat_slice = slice(lat_bound[i], lat_bound[i+1])
        lon_slice = slice(lon_bound[j], lon_bound[j+1])

        z500_train = z500.sel(time=slice('2017', '2017')).isel(lat=lat_slice, lon=lon_slice)['z']
        z500_test = z500.sel(time=slice('2018', '2018.1.7')).isel(lat=lat_slice, lon=lon_slice)['z'] 

        data_mean = z500_train.mean().load()
        data_std = z500_train.std().load()

        # Normalize datasets
        data_train = (z500_train - data_mean) / data_std
        data_test = (z500_test - data_mean) / data_std

        # Training Loop
        lead_time = 24
        space_subsample = 1
        time_subsample = 6 
        frac = 0.1 

        # Train GP model
        model, likelihood, hyperparams_track, X_train, Y_train = train_gp(lead_time, data_train, data_test, space_subsample, time_subsample, frac=frac)

        # Predictions on test set
        Y_pred_actual, Y_test_actual, Y_pred, Y_test, Y_std, lat, lon, wrmse = predictions(model, likelihood, data_train, data_test, data_std, data_mean, lead_time, space_subsample, time_subsample)
        wrmse_full.append(wrmse)

        # Fitting on the training dataset again after training
        model.eval()
        likelihood.eval()
        with torch.no_grad():
            pred_train = model(X_train)
            train_mean = pred_train.mean  
            train_std = pred_train.stddev  

        sorted_idx = torch.argsort(X_train[:,3])

        train_mean = train_mean[sorted_idx].detach().cpu().numpy()
        train_std = train_std[sorted_idx].detach().cpu().numpy()
        Y_train = Y_train[sorted_idx].detach().cpu().numpy()

        # Plots
        rand_coords = X_train.detach().cpu().numpy()
        plot_coords_dist(rand_coords)
        plot_modelparams(hyperparams_track, wrmse, spatial_lengths, period_lengths)
        relative_error = (Y_pred_actual - Y_test_actual) / Y_test_actual * 100 
        relative_error_mean = plot_relative_err(lat, lon, relative_error)
        rmse_error = np.sqrt((Y_pred_actual - Y_test_actual)**2)
        rmse_error_mean = plot_rmse(lat, lon, rmse_error)

        rel_err_idx, rel_err_val = extreme_points_rel_err(relative_error_mean)
        rmse_idx, rmse_val = extreme_points_rmse(rmse_error_mean)
        plot_relative_err_posterior(rel_err_idx, rel_err_val, time_subsample, relative_error_mean, Y_pred, Y_test, Y_std, train_mean, train_std, Y_train)
        plot_rmse_posterior(rmse_idx, rmse_val, time_subsample, rmse_error_mean, Y_pred, Y_test, Y_std, train_mean, train_std, Y_train)

        target_folder = (
            f"plots/5.625deg/z500/Train (2017), test (2018_1_7)/ExactGP_full/lat_{lat_bound[i]}-{lat_bound[i+1]}_lon_{lon_bound[j]}-{lon_bound[j+1]}"
        )
        os.makedirs(target_folder, exist_ok=True)
        tmp_folder = "tmp"
        for files in os.listdir(tmp_folder):
            src_path = os.path.join(tmp_folder, files)
            dst_path = os.path.join(target_folder, files)
            shutil.move(src_path, dst_path)

        full_relative_error_grid[lat_bound[i]:lat_bound[i+1], lon_bound[j]:lon_bound[j+1]] = relative_error_mean
        full_rmse_error_grid[lat_bound[i]:lat_bound[i+1], lon_bound[j]:lon_bound[j+1]] = rmse_error_mean

        del model, likelihood, X_train, Y_train, pred_train, train_mean, train_std, Y_pred, Y_test, Y_std
        torch.cuda.empty_cache()
        gc.collect()

# Full grid     
lat = z500.lat.values
lon = z500.lon.values
plot_relative_err(lat, lon, full_relative_error_grid)
plot_rmse(lat, lon, full_rmse_error_grid)

target_folder_final = "plots/5.625deg/z500/Train (2017), test (2018_1_7)/ExactGP_full/final"
os.makedirs(target_folder_final, exist_ok=True)
tmp_folder = "tmp"
for filename in os.listdir(tmp_folder):
    src_path = os.path.join(tmp_folder, filename)
    dst_path = os.path.join(target_folder_final, filename)
    shutil.move(src_path, dst_path)

print(f"Avg WRMSE: {sum(wrmse_full)/len(wrmse_full)}")